# 04 — Track A, condition A3: semi-supervised adaptation (SemiCLIP) + its no-text control

For each label budget N and seed, two runs of `src/semiclip.py`:

- **A3**: the A2 pairs (119 real expert captions + pseudo-captions of the N labelled images)
  **plus the other ~2,000 Kromp training images without labels** (SemiCLIP: concept pseudo-labels,
  learnable surrogate captions, trapezoidal consistency).
- **A3c** (`--no-text`): the same semi-supervised training with no text anywhere. If A3 beats A2
  but not A3c, the gain comes from the unlabelled images, not from language.

Each run is then re-encoded and scored exactly like A0/A1/A2: linear heads on the same N images
(`src/label_efficiency.py`), plus zero-shot for A3. N = all is skipped (no unlabelled images left).

- **Resumable:** finished runs are saved to Drive and skipped on rerun. If Colab disconnects, `Run all` again.
- **Order:** seed 0 for every N and both conditions first, then seeds 1 and 2.
- Same one-time setup as notebooks 01–03 (Drive data, `GITHUB_TOKEN` secret).

In [ ]:
N_LIST = ['50', '100', '200', '500']
SEEDS = [0, 1, 2]
CONDITIONS = ['A3', 'A3c']      # A3c = --no-text control
EPOCHS1, EPOCHS2 = 20, 5

In [ ]:
# 1. GPU check
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Mount Drive and copy the data to the fast local disk (training re-reads images every step)
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/blastocyst-clip'
LOCAL = '/content/data'
import os
if not os.path.isdir(f'{LOCAL}/blastocyst/Images'):
    !mkdir -p {LOCAL}
    !cp -r {DRIVE}/data/blastocyst {LOCAL}/
    !mkdir -p {LOCAL}/captions && cp -r {DRIVE}/data/captions/images_2 {DRIVE}/data/captions/blastocyst_captions.jsonl {LOCAL}/captions/
print(len(os.listdir(f'{LOCAL}/blastocyst/Images')), 'blastocyst images,',
      len(os.listdir(f'{LOCAL}/captions/images_2')), 'caption images on local disk')

In [ ]:
# 3. Latest code
from google.colab import userdata
REPO = '/content/blastocyst-clip'
token = userdata.get('GITHUB_TOKEN')
url = f'https://{token}@github.com/Daniel0813/blastocyst-clip.git'
if os.path.isdir(REPO):
    !git -C {REPO} pull -q
else:
    !git clone -q {url} {REPO}
del token, url
%cd {REPO}
!git log --oneline -1

In [ ]:
# 4. Install
!pip install -q open_clip_torch
import torch, open_clip, sys, subprocess, json, datetime
sys.path.insert(0, REPO)
print('torch', torch.__version__, '| open_clip', open_clip.__version__, '| cuda', torch.cuda.is_available())
first = json.loads(open(f'{LOCAL}/captions/blastocyst_captions.jsonl').readline())
assert 'holdout' in first, 'caption file lacks the held-out split: run notebook 03 cell 5 first'

In [ ]:
# 5. Smoke test (1 minute): tiny run of both conditions, nothing saved to Drive
!python -W ignore src/semiclip.py --data-dir {LOCAL} --n-labels 50 --limit 8 --epochs1 1 --epochs2 1 --batch-size 4 --out-dir /content/smoke_a3 2>&1 | tail -4
!python -W ignore src/semiclip.py --data-dir {LOCAL} --n-labels 50 --limit 8 --epochs1 1 --epochs2 1 --batch-size 4 --no-text --out-dir /content/smoke_a3c 2>&1 | tail -2

In [ ]:
# 6. The grid (resumable). Results: {DRIVE}/results/label_efficiency/A3_*.csv, A3c_*.csv, A3zs_*.csv
import pandas as pd
from pathlib import Path
from src import zeroshot, evaluate
RES = f'{DRIVE}/results/label_efficiency'
os.makedirs(RES, exist_ok=True)
gold = evaluate.load_gold(f'{LOCAL}/blastocyst/labels.csv')
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD']).decode().strip()

runs = [(n, s, c) for s in SEEDS for n in N_LIST for c in CONDITIONS]
for n, seed, cond in runs:
    name = f'biomedclip_{cond.lower()}_n{n}_seed{seed}'
    done_file = f'{RES}/{cond}_{name}.csv'
    if os.path.exists(done_file):
        print('skip (done):', name); continue
    print(f'\n===== {name}  ({datetime.datetime.now():%H:%M}) =====')
    ft_dir, emb_dir = f'{DRIVE}/results/finetune/{name}', f'/content/emb/{name}'
    flag = '--no-text' if cond == 'A3c' else ''
    !python -W ignore src/semiclip.py --data-dir {LOCAL} --n-labels {n} --seed {seed} {flag} --epochs1 {EPOCHS1} --epochs2 {EPOCHS2} --out-dir {ft_dir} 2>&1 | grep -E "pairs|training|epoch +(1|10|20)/|stage 2 epoch|pseudo-label|saved"
    !python -W ignore src/embed.py --data-dir {LOCAL} --weights {ft_dir}/adapter.pt --out-dir {emb_dir} 2>&1 | grep -E "saved|loaded"
    if cond == 'A3':   # zero-shot with the adapted model (prompt matching, no heads)
        zs = zeroshot.predict(Path(emb_dir))
        zs = zs[zs.image.isin(gold.index)]
        zs_path = f'{emb_dir}/zeroshot_pred.csv'; zs.to_csv(zs_path, header=False, index=False)
        evaluate.evaluate(zs_path, gold).assign(n=n, seed=seed, condition='A3-zeroshot', emb=name) \
            .to_csv(f'{RES}/A3zs_{name}.csv', index=False)
        # TransCLIP with the adapted model's zero-shot as prior (seconds; scored on the test set)
        !python -W ignore src/transclip.py --prior text --emb-dir {emb_dir} --labels {LOCAL}/blastocyst/labels.csv --out-dir {DRIVE}/results/transclip 2>&1 | grep -E "^(TransCLIP|zero-shot)"
    # linear heads on the same N labelled images (written last = marks the run as done)
    !python -W ignore src/label_efficiency.py --condition {cond} --emb-dir {emb_dir} --labels {LOCAL}/blastocyst/labels.csv --n {n} --seeds {seed} --out-dir {RES} 2>&1 | grep -E "{cond} n="
    json.dump({'commit': commit, 'date': datetime.datetime.now().isoformat(timespec='seconds'),
               'gpu': torch.cuda.get_device_name(0)}, open(f'{ft_dir}/run_info.json', 'w'))
    !rm -rf {emb_dir}
print('\nall runs done' if all(os.path.exists(f"{RES}/{c}_biomedclip_{c.lower()}_n{n}_seed{s}.csv") for n, s, c in runs) else '')

When finished (or whenever you want a look at partial results), download from Drive into the
local project's `results/` folder:
- `MyDrive/blastocyst-clip/results/label_efficiency/` (new files `A3_*`, `A3c_*`, `A3zs_*`, `predictions/`)
- `MyDrive/blastocyst-clip/results/transclip/` (TransCLIP on each A3 model's zero-shot)
- `MyDrive/blastocyst-clip/results/finetune/biomedclip_a3*/config.json` and `pseudo_labels.csv`
  (training curves and pseudo-label accuracy; the `.pt` files can stay on Drive)

Then locally: `python src/plot_label_efficiency.py`.